# Trend Agent

Runs the full pipeline: **Goal -> Plan -> Retrieve -> Analyze -> Decide -> Recommend -> Validate**.

In [1]:
import sys
from pathlib import Path

import faiss
import pandas as pd
from sentence_transformers import SentenceTransformer

def _locate_and_import_agent_core():
    here = Path.cwd()
    candidates = [here, here / "agents", here.parent / "agents", here.parent]
    for c in candidates:
        if (c / "agent_core.py").exists():
            sys.path.insert(0, str(c))
            import agent_core as agent_core_module
            return agent_core_module
    raise FileNotFoundError(
        "Could not locate agent_core.py. Looked in: "
        + ", ".join(str(c) for c in candidates)
        + f". Current working directory: {here}"
    )


core = _locate_and_import_agent_core()

DATA_DIR = core.find_data_dir()
CHUNKS_PATH   = DATA_DIR / "chunked_data.json"
INDEX_PATH    = DATA_DIR / "sap_intelligence.index"
OUT_PATH      = DATA_DIR / "trends.json"
PIPELINE_PATH = DATA_DIR / "trends_pipeline.json"

EMBED_MODEL = "BAAI/bge-small-en-v1.5"

GOAL = (
    "Identify 3 to 5 emerging trends relevant to SAP - technology "
    "trends, customer behaviour shifts, or industry developments - "
    "using the indexed SAP news corpus as evidence."
)

print("data dir:", DATA_DIR)


data dir: /home/jovyan/vault/Testing/Testing/notebook/data


In [2]:
chunks_df = pd.read_json(CHUNKS_PATH)
index = faiss.read_index(str(INDEX_PATH))
embed_model = SentenceTransformer(EMBED_MODEL)

print("chunks loaded:", len(chunks_df))
print("vectors in index:", index.ntotal)

core.warmup()


chunks loaded: 1400
vectors in index: 1400
warming up model...


warmup done in 0.5 sec


In [3]:
def retrieve_news(query, k=6):
    q_vec = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    _, idx = index.search(q_vec, k)
    return chunks_df.iloc[idx[0]]["text"].tolist()


def read_previous_findings(_args=None):
    previous = core.load_previous_output(OUT_PATH)
    if not previous:
        return "No previous findings on file - this looks like the first run."
    return previous


RETRIEVAL_TOOL_HANDLERS = {
    "retrieve_news": lambda args: retrieve_news(args.get("query", ""), int(args.get("k", 6) or 6)),
    "read_previous_findings": lambda args: read_previous_findings(args),
}

RETRIEVAL_TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "retrieve_news",
            "description": (
                "Semantic search over the indexed SAP news corpus. Call this "
                "with your own queries to investigate different angles - "
                "technology trends, customer behaviour shifts, industry "
                "developments, etc."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Natural language search query"},
                    "k": {"type": "integer", "description": "Number of chunks to retrieve (default 6)"},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_previous_findings",
            "description": (
                "Read the trends you identified in the previous run, if any, "
                "so you can check what is still supported by current evidence."
            ),
            "parameters": {"type": "object", "properties": {}},
        },
    },
]


In [4]:
DECIDE_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "finalize_trends",
        "description": "Call this exactly once to submit your final list of trends.",
        "parameters": {
            "type": "object",
            "properties": {
                "trends": {
                    "type": "array",
                    "description": "3 to 5 emerging trends",
                    "items": {
                        "type": "object",
                        "properties": {
                            "title": {"type": "string"},
                            "category": {"type": "string", "description": "Technology, Customer Behaviour, or Industry"},
                            "description": {"type": "string", "description": "1-2 sentences explaining the trend"},
                            "evidence": {"type": "string"},
                        },
                        "required": ["title", "category", "description", "evidence"],
                    },
                }
            },
            "required": ["trends"],
        },
    },
}

VALIDATE_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "submit_validated_trends",
        "description": "Call this exactly once to submit your evidence-checked, final list of trends.",
        "parameters": {
            "type": "object",
            "properties": {
                "validated_trends": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "title": {"type": "string"},
                            "category": {"type": "string"},
                            "description": {"type": "string"},
                            "evidence": {"type": "string"},
                            "validation_status": {"type": "string", "description": "Supported, Revised, or Unsupported"},
                            "validation_notes": {"type": "string"},
                        },
                        "required": ["title", "category", "description", "evidence", "validation_status", "validation_notes"],
                    },
                }
            },
            "required": ["validated_trends"],
        },
    },
}


In [5]:
validated_trends, pipeline = core.run_full_pipeline(
    goal_description=GOAL,
    retrieval_tool_schemas=RETRIEVAL_TOOL_SCHEMAS,
    retrieval_tool_handlers=RETRIEVAL_TOOL_HANDLERS,
    analyze_tool_schema=core.make_analyze_tool_schema("trends"),
    decide_tool_schema=DECIDE_TOOL_SCHEMA,
    decide_tool_name="finalize_trends",
    decide_items_key="trends",
    validate_tool_schema=VALIDATE_TOOL_SCHEMA,
    validate_tool_name="submit_validated_trends",
    validate_items_key="validated_trends",
)

print()
print("=" * 70)
print(len(validated_trends), "validated trends:")
for t in validated_trends:
    print("-", t.get("title"), "|", t.get("category"), "|", t.get("validation_status"))



STAGE 1: PLAN


PLAN: {
  "goal": "Identify emerging trends relevant to SAP by analyzing the latest news and developments in the field.",
  "planned_steps": [
    "Search the indexed SAP news corpus for recent articles on technology advancements",
    "Analyze customer behaviour shifts as reported in industry publications and blogs",
    "Consult industry reports and research studies to identify key trends",
    "Check online forums and discussion boards for relevant conversations",
    "Use natural language processing techniques to categorize and prioritize emerging trends"
  ]
}

STAGE 2: RETRIEVE


  [retrieve step 1] called `retrieve_news` with {"k": "6", "query": "latest SAP technology advancements"}


  [retrieve step 2] no tool call - treating retrieval as complete.

STAGE 3: ANALYZE


ANALYSIS: {
  "candidate_items": "[\"SAP AI innovations\", \"Next-Gen SAP Ariba\", \"Cloud ALM\"]",
  "key_observations": [
    "The team continues to innovate with AI that learns instantly from user feedback, \\u201csees\\u201d and interprets documents visually, and understands content well enough to take intelligent actions on it.",
    "Next generation of generative AI models now support over 110 languages.",
    "Building on these innovations, platform usage on SAP BTP for custom document automation has increased 285-fold since 2020."
  ]
}

STAGE 4: DECIDE + RECOMMEND


DRAFT DECISION: {
  "trends": "[{\"category\": \"SAP AI innovations\", \"title\": \"AI that learns instantly from user feedback and interprets documents visually\", \"description\": \"The team continues to innovate with AI that learns instantly from user feedback, \\u201csees\\u201d and interprets documents visually, and understands content well enough to take intelligent actions on it.\", \"evidence\": \"Next-generation of generative AI models now support over 110 languages.\"}, {\"category\": \"SAP Business Network\", \"title\": \"AI-infused orchestration in manufacturing\", \"description\": \"Manufacturers create ESPR-ready product records capturing environmental impact, material composition, repairability, and recyclability data.\", \"evidence\": \"General availability is planned for Q2 2026.\"}, {\"category\": \"Next-Gen SAP Ariba\", \"title\": \"AI-driven procurement innovation\", \"description\": \"Building on these innovations, platform usage on SAP BTP for custom document auto

VALIDATED: {
  "validated_trends": "[{\"category\": \"SAP AI innovations\", \"title\": \"AI that learns instantly from user feedback and interprets documents visually\", \"description\": \"The team continues to innovate with AI that learns instantly from user feedback, \\u201csees\\u201d and interprets documents visually, and understands content well enough to take intelligent actions on it.\", \"evidence\": \"Next-generation of generative AI models now support over 110 languages.\", \"validation_status\": \"Supported\", \"validation_notes\": \"\"}, {\"category\": \"SAP Business Network\", \"title\": \"AI-infused orchestration in manufacturing\", \"description\": \"Manufacturers create ESPR-ready product records capturing environmental impact, material composition, repairability, and recyclability data.\", \"evidence\": \"General availability is planned for Q2 2026.\", \"validation_status\": \"Supported\", \"validation_notes\": \"\"}, {\"category\": \"Next-Gen SAP Ariba\", \"title\": \

In [6]:
core.save_json(OUT_PATH, validated_trends)
core.save_json(PIPELINE_PATH, pipeline)


saved to /home/jovyan/vault/Testing/Testing/notebook/data/trends.json
saved to /home/jovyan/vault/Testing/Testing/notebook/data/trends_pipeline.json
